# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides an example for loading and exploring the FAIR² dataset using the `mlcroissant` library. All dataset entities (record sets, fields, columns) are referenced by their `@id` values. You can apply this workflow to other Croissant packages too.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset Croissant metadata and initialize the Dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant package
dataset = mlc.Dataset(croissant_url)

# Access and view dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

List the available record sets (by `@id`), and fields (by their `@id`), then provide basic information for each.

In [ ]:
# List all Record Sets by their @id
print("Available Record Sets:")
for rs in dataset.record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}, fields: {[f.id for f in rs.fields]}")

View sample records for each RecordSet using their `@id`. Here, we view the records in the main record set.

In [ ]:
# Example: List the first 3 records from the main record set

# Identify the main record set @id (use the first one by default)
main_record_set = dataset.record_sets[0]
main_record_set_id = main_record_set.id

print(f"\nFirst three records in record set @id={main_record_set_id}:")
for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    if i >= 3:
        break
    print(record)


## 3. Data Extraction

Load all records from each record set into a pandas DataFrame for further exploration. All lookups use the `@id` of the entities.

In [ ]:
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    # List of records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show columns of the main record set's DataFrame, using its @id
print(f"\nDataFrame columns for record set @id={main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())

# Show first 5 records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

You can now process and explore the dataset. As an example, we'll:
- Filter records on a numeric field (e.g., patient age)
- Normalize that field
- Group by a categorical field, e.g., anatomical site

All field names and columns are accessed using their `@id`.

In [ ]:
# You may need to adjust these @id values based on the actual field @id's; here we try common names

# Inspect DataFrame columns (@id's) for the main record set
cols = dataframes[main_record_set_id].columns.tolist()
print("Available columns (@id):", cols)

# Let's try to find an age column, or use any numeric field as an example
# Common naming: try 'age', else use the first float/integer column
numeric_field_id = None
for c in cols:
    if 'age' in c.lower():
        numeric_field_id = c
        break
if numeric_field_id is None:
    # fallback: select first numeric-looking column
    for c in cols:
        if dataframes[main_record_set_id][c].dtype in [int, float, 'int64', 'float64']:
            numeric_field_id = c
            break

# If no numeric field found, skip EDA step
if numeric_field_id is not None:
    print(f"\nUsing numeric field @id: {numeric_field_id}")

    # Filter for values > threshold (choose a sensible threshold, e.g., age>40 or value>10):
    threshold = 40
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]

    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field
    group_field_id = None
    # Search for the first column that is string/object type and not the numeric field
    for c in cols:
        if c == numeric_field_id:
            continue
        if dataframes[main_record_set_id][c].dtype == object:
            group_field_id = c
            break

    if group_field_id is not None:
        print(f"\nGrouping by field @id: {group_field_id}")
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        )
        print(grouped_df.head())
else:
    print("No numeric field found for analysis.")

## 5. Visualization

Let's visualize the distribution of the numeric field, and the group-wise mean if grouping is available. You may need to install matplotlib for visualization.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    dataframes[main_record_set_id][numeric_field_id].hist(bins=15, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Plot group-wise mean if grouped_df exists
    if 'grouped_df' in locals() and group_field_id is not None:
        plt.figure(figsize=(10,5))
        plt.bar(grouped_df[group_field_id], grouped_df[numeric_field_id], color='salmon')
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to use `mlcroissant` to access and analyze a FAIR² Croissant-structured clinical dataset. By referencing each entity using its `@id`, we ensured robust and transparent data handling. You can adapt these techniques to filter, aggregate, and visualize structured datasets provided as Croissant schemas for your own research.
